# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Agha314/FLyRank-Task-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:**
The original table, `fact_content_daily_performance`, has one row for each day, page, and client. In other words, each row represents the performance of one page for one client on one specific day.

For the model, I combine these daily records into one row for each page and client for the whole month. So, the model works with **one page per client for March 2026**.

**Time Window:**
The data used is from **March 2026 (March 1 to March 31)**. This month was chosen because it is in the middle of the available data period.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


Features (Honest X), aggregated per page over March:
- total_gsc_impressions_mar = SUM(gsc_impressions) — a count of past search impressions,
  fully known by month-end.
- total_gsc_clicks_mar = SUM(gsc_clicks) — past clicks, same reasoning.
- avg_gsc_position_mar = AVG(gsc_avg_position) — historical average ranking position
  over the month, known at decision time.
- ctr_mar = total_gsc_clicks_mar / total_gsc_impressions_mar — derived from the two
  historical counts above, computable at month-end.
- days_gsc_available = COUNT(*) WHERE gsc_data_available IS TRUE — how many days of
  trustworthy GSC data this page actually had in March, known at decision time.

Label (Proxy Y): is_declining. Split March into first half (days 1–15) and second half
(days 16–31); sum gsc_clicks per page in each half. is_declining = 1 if second-half
clicks < first-half clicks, else 0. This is a within-month proxy, not a true forward-
looking trend — a real trend label needs multiple months, which this single partition
doesn't have (noted in Section 4).

Excluded: second_half_clicks (and any other second-half-only aggregate).

Why: this is
the exact quantity the label is computed from — including it as a feature would let the
model see the answer directly.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_Token')}')")

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
con.sql(f'SELECT column_name FROM user_tab_columns WHERE table_name = {REL} ORDER BY column_id;')

# # con.sql(f"DESCRIBE SELECT * FROM {REL}").show()
# con.sql(f"SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date FROM {REL}").show()

# con.sql(f'SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n FROM {REL} GROUP BY report_date, client_hash_id, content_hash_id HAVING COUNT(*) > 1 LIMIT 5')
# con.sql(f'SELECT COUNT(*) AS n_available FROM {REL} WHERE gsc_data_available IS TRUE')

CatalogException: Catalog Error: Table with name user_tab_columns does not exist!
Did you mean "duckdb_columns"?

In [11]:
res = con.execute(f"SELECT * FROM {REL} LIMIT 5")
dlt=res.df()
dlt

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [7]:
result = con.execute(f"DESCRIBE SELECT * FROM {REL}").fetchall()
count=0
for row in result:
    print(row)
    count=count+1

print(count)

('report_date', 'DATE', 'YES', None, None, None)
('client_hash_id', 'VARCHAR', 'YES', None, None, None)
('content_hash_id', 'VARCHAR', 'YES', None, None, None)
('client_has_gsc', 'BOOLEAN', 'YES', None, None, None)
('client_has_ga4', 'BOOLEAN', 'YES', None, None, None)
('gsc_data_available', 'BOOLEAN', 'YES', None, None, None)
('ga4_data_available', 'BOOLEAN', 'YES', None, None, None)
('gsc_impressions', 'BIGINT', 'YES', None, None, None)
('gsc_clicks', 'BIGINT', 'YES', None, None, None)
('gsc_sum_position', 'BIGINT', 'YES', None, None, None)
('gsc_avg_position', 'DOUBLE', 'YES', None, None, None)
('ga4_pageviews', 'BIGINT', 'YES', None, None, None)
('ga4_sessions', 'BIGINT', 'YES', None, None, None)
('ga4_users', 'BIGINT', 'YES', None, None, None)
('ga4_engaged_sessions', 'BIGINT', 'YES', None, None, None)
('ga4_total_engagement_sec', 'BIGINT', 'YES', None, None, None)
('sessions_organic', 'BIGINT', 'YES', None, None, None)
('sessions_direct', 'BIGINT', 'YES', None, None, None)
('sess

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.